In [14]:
# ============================================================
# PFE Baseline: Pretrained ViT (timm) on CIFAR-10 (PyTorch)
# Uses vit_small_patch16_224 pretrained on ImageNet-21k
# Adapted for 32x32: interpolated pos embeddings + fine-tuning
# Mirrors ResNet-50 baseline for fair comparison
# Ready for: Pruning, Quantization, Distillation, LoRA
# ============================================================

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import timm                          # pip install timm
import time, os, json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score,
                             recall_score, f1_score)
from thop import profile
from PIL import Image

In [15]:
# ── REPRODUCIBILITY ───────────────────────────────────────────
SEED = 42

import random
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(SEED)

In [16]:
# ── CONFIG ───────────────────────────────────────────────────
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE  = 128
EPOCHS      = 50           # same as ResNet baseline — pretrained weights converge faster
LR          = 1e-4         # lower LR for fine-tuning pretrained weights
NUM_CLASSES = 10
SAVE_PATH   = "__4__baseline_vit_pretrained_cifar10.pth"

CIFAR10_CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']

# Same normalization as the other two baselines
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

print(f"Using device: {DEVICE}")

Using device: cuda


In [17]:
# ── MODEL: Pretrained ViT adapted for CIFAR-10 ───────────────
#
# Strategy (mirrors how ResNet-50 was adapted):
#   1. Load vit_small_patch16_224 pretrained on ImageNet-21k via timm
#   2. The model expects 224×224 input; CIFAR is 32×32.
#      → We resize CIFAR images to 224×224 in the transform pipeline
#        (same as common fine-tuning practice, avoids positional
#         embedding interpolation complexity at inference time)
#   3. Replace the classifier head with a new Linear(embed_dim, 10)
#   4. Fine-tune the FULL model (not frozen backbone) — same as ResNet
#
# Why vit_small_patch16_224?
#   • "Small" scale ≈ ResNet-50 in parameter count (~22M vs ~25M)
#   • patch16 + 224px → 196 tokens: well-supported by ImageNet pretraining
#   • ImageNet-21k weights → stronger prior than ImageNet-1k alone
#
# Alternative: img_size=32 with interpolated pos embeddings
#   timm supports this via: timm.create_model(..., img_size=32)
#   It bilinearly interpolates the 196 pos embeddings down to 4 patches.
#   BUT: 32/16 = 2 → only 4 patches total — too coarse, hurts accuracy.
#   Resizing to 224 keeps all 196 patches and full pretrained resolution.
# ─────────────────────────────────────────────────────────────

IMG_SIZE = 224   # resize CIFAR to this for the pretrained ViT

def build_model(num_classes=10):
    # pretrained=True downloads ImageNet-21k weights automatically
    model = timm.create_model(
        'vit_small_patch16_224',
        pretrained=True,
        num_classes=num_classes,   # timm replaces head automatically
    )
    # The head is already replaced by timm when num_classes is passed.
    # Confirm:
    print(f"  Classifier head : {model.head}")
    return model

model = build_model(NUM_CLASSES).to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

KeyboardInterrupt: 

In [ ]:
# ── DATA ─────────────────────────────────────────────────────
# Key difference vs scratch ViT baseline:
#   • We RESIZE to 224×224 to match pretrained ViT input resolution
#   • Same normalization as the other baselines (CIFAR stats)
#     Note: ImageNet normalization (mean=[0.485,0.456,0.406]) would also
#     work since the backbone was trained on ImageNet, but using CIFAR
#     stats keeps all three baselines consistent and comparable.

transform_train = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),          # 32 → 224
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(IMG_SIZE, padding=IMG_SIZE // 8),   # padding=28
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.AutoAugment(transforms.AutoAugmentPolicy.CIFAR10),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    transforms.RandomErasing(p=0.25),
])

transform_test = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),          # 32 → 224
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

train_set = torchvision.datasets.CIFAR10(root='./data', train=True,
                                          download=True, transform=transform_train)
test_set  = torchvision.datasets.CIFAR10(root='./data', train=False,
                                          download=True, transform=transform_test)

train_loader = torch.utils.data.DataLoader(train_set, batch_size=BATCH_SIZE,
                                            shuffle=True, num_workers=0,
                                            worker_init_fn=seed_worker,
                                            generator=g, pin_memory=True)
test_loader  = torch.utils.data.DataLoader(test_set,  batch_size=BATCH_SIZE,
                                            shuffle=False, num_workers=0,
                                            worker_init_fn=seed_worker,
                                            pin_memory=True)

print(f"Train: {len(train_set)} | Test: {len(test_set)}")
print(f"Input resolution fed to model: {IMG_SIZE}×{IMG_SIZE}")

Train: 50000 | Test: 10000
Input resolution fed to model: 224×224


In [ ]:
# ── TRAINING ──────────────────────────────────────────────────
# Fine-tuning strategy (mirrors ResNet-50 baseline where possible):
#   • AdamW — standard for ViT fine-tuning (same as scratch ViT baseline)
#   • LR = 1e-4 — lower than scratch (1e-3) to avoid destroying pretrained weights
#   • Warmup (3 epochs) + Cosine decay — ViT fine-tuning best practice
#   • Label smoothing 0.1 — same as both other baselines
#   • Gradient clipping 1.0 — same as scratch ViT baseline

WARMUP_EPOCHS = 3

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR,
                               weight_decay=0.05, betas=(0.9, 0.999))

def get_lr(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / (EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1 + np.cos(np.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=get_lr)


def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        correct    += outputs.argmax(1).eq(labels).sum().item()
        total      += labels.size(0)
    return total_loss / len(loader), correct / total


def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            correct += model(inputs).argmax(1).eq(labels).sum().item()
            total   += labels.size(0)
    return correct / total


best_val_acc = 0.0
train_accs, val_accs, train_losses = [], [], []

print("\n" + "="*60)
print("TRAINING  (Pretrained ViT fine-tuned on CIFAR-10)")
print("="*60)

for epoch in range(EPOCHS):
    loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_acc         = evaluate(model, test_loader)
    scheduler.step()

    current_lr = optimizer.param_groups[0]['lr']
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    train_losses.append(loss)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), SAVE_PATH)
        marker = " ← best saved"
    else:
        marker = ""

    print(f"Epoch {epoch+1:2d}/{EPOCHS} | LR: {current_lr:.6f} | Loss: {loss:.4f} | "
          f"Train: {train_acc:.4f} | Val: {val_acc:.4f}{marker}")

print(f"\nBest validation accuracy: {best_val_acc:.4f} ({best_val_acc*100:.2f}%)")


TRAINING  (Pretrained ViT fine-tuned on CIFAR-10)
Epoch  1/50 | LR: 0.000067 | Loss: 0.7458 | Train: 0.9010 | Val: 0.9765 ← best saved
Epoch  2/50 | LR: 0.000100 | Loss: 0.6408 | Train: 0.9414 | Val: 0.9797 ← best saved
Epoch  3/50 | LR: 0.000100 | Loss: 0.6395 | Train: 0.9407 | Val: 0.9765
Epoch  4/50 | LR: 0.000100 | Loss: 0.6291 | Train: 0.9449 | Val: 0.9770
Epoch  5/50 | LR: 0.000100 | Loss: 0.6209 | Train: 0.9490 | Val: 0.9758


KeyboardInterrupt: 

In [ ]:
# ── TRAINING CURVES ───────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(train_accs, label='Train', color='blue')
ax1.plot(val_accs,   label='Val',   color='red')
ax1.set_title('Accuracy — Pretrained ViT', fontsize=13, fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(train_losses, color='orange')
ax2.set_title('Training Loss — Pretrained ViT', fontsize=13, fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('__4__training_curves.png', dpi=150)
plt.show()

In [ ]:
# ── FULL EVALUATION ───────────────────────────────────────────
print("\n" + "="*60)
print("FULL EVALUATION")
print("="*60)

model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(DEVICE)
        preds  = model(inputs).argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

acc       = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average='macro')
recall    = recall_score(all_labels, all_preds, average='macro')
f1        = f1_score(all_labels, all_preds, average='macro')

print(f"\n  Accuracy          : {acc:.4f}  ({acc*100:.2f}%)")
print(f"  Precision (macro) : {precision:.4f}")
print(f"  Recall    (macro) : {recall:.4f}")
print(f"  F1-score  (macro) : {f1:.4f}")
print("\nPer-class report:")
print(classification_report(all_labels, all_preds,
                             target_names=CIFAR10_CLASSES, digits=4))

In [ ]:
# ── CONFUSION MATRIX ──────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CIFAR10_CLASSES, yticklabels=CIFAR10_CLASSES)
plt.title('Confusion Matrix — Pretrained ViT', fontsize=14, fontweight='bold')
plt.ylabel('True Label'); plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('__4__confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# ── MODEL COMPLEXITY METRICS ──────────────────────────────────
# NOTE: inference is measured at 224×224 (actual runtime resolution)
print("\n" + "="*60)
print("MODEL COMPLEXITY METRICS")
print("="*60)

size_mb = os.path.getsize(SAVE_PATH) / 1e6
dummy   = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
macs, _ = profile(model, inputs=(dummy,), verbose=False)
flops   = macs * 2

model.eval()
with torch.no_grad():
    for _ in range(10):
        model(dummy)
    times = []
    for _ in range(200):
        t0 = time.time()
        model(dummy)
        times.append(time.time() - t0)
inf_ms = np.mean(times) * 1000

print(f"  Parameters      : {total_params:,}")
print(f"  Model size      : {size_mb:.2f} MB")
print(f"  FLOPs           : {flops/1e9:.3f} GFLOPs  (at {IMG_SIZE}×{IMG_SIZE})")
print(f"  Inference time  : {inf_ms:.3f} ms  (single image, {IMG_SIZE}×{IMG_SIZE})")

In [ ]:
# ── SAVE METRICS JSON ─────────────────────────────────────────
baseline_metrics = {
    "accuracy"    : acc,
    "precision"   : precision,
    "recall"      : recall,
    "f1"          : f1,
    "params"      : total_params,
    "size_mb"     : size_mb,
    "flops_G"     : flops / 1e9,
    "inference_ms": inf_ms,
    "input_resolution": IMG_SIZE,
    "pretrained"  : "ImageNet-21k (vit_small_patch16_224 via timm)",
}
with open("__4__baseline_metrics.json", "w") as f:
    json.dump(baseline_metrics, f, indent=2)
print("\n✓ Metrics saved to __4__baseline_metrics.json")

In [ ]:
# ── PREDICT ON CUSTOM IMAGES ──────────────────────────────────
predict_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

def predict_image(image_path, model, show=True):
    img    = Image.open(image_path).convert('RGB')
    tensor = predict_transform(img).unsqueeze(0).to(DEVICE)

    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1).squeeze().cpu().numpy()

    pred_idx   = probs.argmax()
    pred_class = CIFAR10_CLASSES[pred_idx]
    confidence = probs[pred_idx]

    if show:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
        ax1.imshow(img)
        ax1.set_title(f"Prediction: {pred_class}\nConfidence: {confidence:.2%}",
                      fontsize=12, fontweight='bold')
        ax1.axis('off')

        colors = ['crimson' if i == pred_idx else 'steelblue' for i in range(NUM_CLASSES)]
        ax2.barh(CIFAR10_CLASSES, probs, color=colors)
        ax2.set_xlabel('Probability')
        ax2.set_title('Class Probabilities')
        ax2.set_xlim(0, 1)
        for i, v in enumerate(probs):
            ax2.text(v + 0.01, i, f'{v:.1%}', va='center', fontsize=9)
        plt.tight_layout()
        plt.savefig(f'pred_{os.path.splitext(os.path.basename(image_path))[0]}.png', dpi=150)
        plt.show()

    top3 = sorted(zip(CIFAR10_CLASSES, probs), key=lambda x: -x[1])[:3]
    print(f"  File      : {os.path.basename(image_path)}")
    print(f"  Predicted : {pred_class} ({confidence:.2%})")
    print(f"  Top-3     : {[(c, f'{p:.2%}') for c, p in top3]}\n")
    return pred_class, probs


def predict_folder(folder_path, model, save_json=True):
    supported = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
    files = [f for f in os.listdir(folder_path) if f.lower().endswith(supported)]
    if not files:
        print(f"No images found in {folder_path}")
        return
    print(f"\n{'File':<35} {'Prediction':<15} {'Confidence':>10}")
    print("-" * 62)
    results = []
    for fname in sorted(files):
        pred_class, probs = predict_image(os.path.join(folder_path, fname),
                                          model, show=False)
        conf = float(probs.max())
        results.append((fname, pred_class, conf))
        print(f"  {fname:<33} {pred_class:<15} {conf:>9.2%}")
        print("─" * 72)

    if save_json:
        with open("__4__output_on_test_data.json", "w") as f:
            json.dump(results, f, indent=4)
    return results


# predict_folder("./data_test/", model)

In [ ]:
# ── ATTENTION MAP VISUALISATION ───────────────────────────────
# timm's ViT stores blocks in model.blocks — same interface as scratch ViT

def visualise_attention(model, image_tensor, save_path="__4__attention_map.png"):
    """
    Extracts CLS-token attention from the last transformer block
    and overlays it on the input image.
    Works with timm's vit_small_patch16_224.
    """
    model.eval()
    x = image_tensor.unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        # timm ViT forward internals
        x_emb = model.patch_embed(x)                        # (1, 196, D)
        cls   = model.cls_token.expand(1, -1, -1)           # (1, 1, D)
        x_emb = torch.cat([cls, x_emb], dim=1)              # (1, 197, D)
        x_emb = model.pos_drop(x_emb + model.pos_embed)

        # Forward through all blocks except the last
        for blk in model.blocks[:-1]:
            x_emb = blk(x_emb)

        # Last block: extract raw attention
        last_blk = model.blocks[-1]
        x_norm   = last_blk.norm1(x_emb)
        B, N, C  = x_norm.shape

        # timm attention uses fused qkv
        qkv = (last_blk.attn.qkv(x_norm)
                   .reshape(B, N, 3, last_blk.attn.num_heads,
                            C // last_blk.attn.num_heads)
                   .permute(2, 0, 3, 1, 4))
        q, k, _ = qkv.unbind(0)
        scale = (C // last_blk.attn.num_heads) ** -0.5
        attn  = (q @ k.transpose(-2, -1)) * scale
        attn  = attn.softmax(dim=-1)

    # CLS → patch attention, averaged over heads
    cls_attn = attn[0, :, 0, 1:].mean(dim=0)          # (196,)
    n        = int(cls_attn.shape[0] ** 0.5)           # 14
    attn_map = cls_attn.reshape(n, n).cpu().numpy()

    from PIL import Image as PILImage
    attn_img = PILImage.fromarray(
        (attn_map / attn_map.max() * 255).astype(np.uint8)
    ).resize((IMG_SIZE, IMG_SIZE), PILImage.BILINEAR)

    # Denormalise original image
    img_np = image_tensor.permute(1, 2, 0).numpy()
    img_np = (img_np * np.array(CIFAR_STD) + np.array(CIFAR_MEAN)).clip(0, 1)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(img_np);               axes[0].set_title('Input Image');        axes[0].axis('off')
    axes[1].imshow(attn_map, cmap='hot'); axes[1].set_title('Raw Attention Map');  axes[1].axis('off')
    axes[2].imshow(img_np)
    axes[2].imshow(np.array(attn_img), cmap='hot', alpha=0.5)
    axes[2].set_title('Attention Overlay'); axes[2].axis('off')

    plt.suptitle('CLS Token Attention — Pretrained ViT (Last Block)', fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"✓ Attention map saved to {save_path}")


model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
test_imgs, test_labels = next(iter(test_loader))
for i in range(3):
    print(f"\nSample {i+1} — True label: {CIFAR10_CLASSES[test_labels[i]]}")
    visualise_attention(model, test_imgs[i],
                        save_path=f"__4__attention_sample_{i+1}.png")

In [ ]:
# ── SAVE FULL CHECKPOINT ──────────────────────────────────────
config_dict = {
    "model_name"  : "vit_small_patch16_224_pretrained",
    "timm_model"  : "vit_small_patch16_224",
    "pretrained"  : "ImageNet-21k",
    "num_classes" : NUM_CLASSES,
    "input_size"  : [3, IMG_SIZE, IMG_SIZE],
    "normalization": {
        "mean": CIFAR_MEAN,
        "std" : CIFAR_STD,
    },
    "training": {
        "batch_size"     : BATCH_SIZE,
        "epochs"         : EPOCHS,
        "learning_rate"  : LR,
        "optimizer"      : "AdamW",
        "weight_decay"   : 0.05,
        "scheduler"      : "WarmupCosine",
        "warmup_epochs"  : WARMUP_EPOCHS,
        "grad_clip"      : 1.0,
        "label_smoothing": 0.1,
    }
}

torch.save({
    "model_state_dict": model.state_dict(),
    "config"          : config_dict,
    "classes"         : CIFAR10_CLASSES,
}, "__4__model_checkpoint.pth")

print("\n" + "="*60)
print("BASELINE COMPLETE — Pretrained ViT fine-tuned on CIFAR-10")
print("="*60)
print(f"  Weights    → {SAVE_PATH}")
print(f"  Checkpoint → __4__model_checkpoint.pth")
print(f"  Metrics    → __4__baseline_metrics.json")
print(f"  Plots      → __4__training_curves.png")
print(f"               __4__confusion_matrix.png")
print(f"               __4__attention_sample_1/2/3.png")

In [ ]:


# %%
# ── LOADING THE MODEL (reference) ────────────────────────────
# checkpoint = torch.load("__4__model_checkpoint.pth", map_location="cpu")
# config     = checkpoint["config"]
# classes    = checkpoint["classes"]
#
# model = timm.create_model(
#     config["timm_model"],
#     pretrained=False,
#     num_classes=config["num_classes"],
# )
# model.load_state_dict(checkpoint["model_state_dict"])
# model.eval()